### Multimodal RAG (PDF With Images)



In [41]:
import os
import io
import base64  #(required format for image input to CLIP)

import fitz # PyMuPDF(read pdf) 
import numpy as np
import torch

from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

from transformers import CLIPProcessor, CLIPModel

from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [42]:
###Clip Model(Main aim is to convert extracted images and text into embeddings) 

# (CLIP -> Contrastive Language-Image Pre-training)(open source model by OpenAI) is a neural network trained on a variety of (image, text) pairs. It can be used to generate embeddings for both images and text, allowing for tasks such as image classification, zero-shot learning, and cross-modal retrieval.)

import os
from dotenv import load_dotenv
load_dotenv()

## set up the environment for OpenAi Mulitmodel llm
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")#(processor is required as whatever format is required for this process it will convert it into that format only )

clip_model.eval() #shows evaluation of the model (no training is required as we are using pretrained model)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22965.73it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [ ]:

import torch.nn.functional as F

def embed_image(image_data):
    """Generate a normalized CLIP image embedding."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data.convert("RGB")

    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        features = clip_model.visual_projection(outputs.pooler_output)
        features = F.normalize(features, p=2, dim=-1)

    return features.squeeze(0).cpu().numpy()


def embed_text(text):
    """Generate a normalized CLIP text embedding."""
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )

    with torch.no_grad():
        outputs = clip_model.text_model(**inputs)
        features = clip_model.text_projection(outputs.pooler_output)
        features = F.normalize(features, p=2, dim=-1)

    return features.squeeze(0).cpu().numpy()

In [ ]:
text_embedding = embed_text("What is a large language model?")
print(text_embedding.shape)
print(np.linalg.norm(text_embedding))

(512,)
1.0


In [44]:
## Process PDF
pdf_path="docrag_testing_document_by_gpt.pdf"

doc=fitz.open(pdf_path)

# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)




In [ ]:
print("Documents:", len(all_docs))
print("Embeddings:", len(all_embeddings))

print("Text embedding shape:",
      embed_text("What is a large language model?").shape)

print("Image embedding shape:",
      embed_image(Image.new("RGB", (224, 224), "white")).shape)

print("All embeddings same dimension:",
      all(e.shape == (512,) for e in all_embeddings))

Documents: 0
Embeddings: 0
Text embedding shape: (512,)
Image embedding shape: (512,)
All embeddings same dimension: True


In [ ]:
assert len(all_docs) == len(all_embeddings), \
    "Mismatch between documents and embeddings!"

assert all(embedding.shape == (512,) for embedding in all_embeddings), \
    "Invalid embedding dimensions!"

print("All checks passed. Ready to build the FAISS index.")

All checks passed. Ready to build the FAISS index.


In [55]:
import fitz
import os

pdf_path = "docrag_testing_document_by_gpt.pdf"

print("PDF exists:", os.path.exists(pdf_path))

doc = fitz.open(pdf_path)
print("Total pages:", len(doc))

for i, page in enumerate(doc):
    text = page.get_text()
    images = page.get_images(full=True)

    print(
        f"Page {i}: text_chars={len(text.strip())}, "
        f"embedded_images={len(images)}"
    )

doc.close()

PDF exists: True
Total pages: 2
Page 0: text_chars=1679, embedded_images=0
Page 1: text_chars=804, embedded_images=0


In [56]:
doc

closed Document('docrag_testing_document_by_gpt.pdf')

In [57]:
all_docs = []
all_embeddings = []
image_data_store = {}

doc = fitz.open(pdf_path)

for i, page in enumerate(doc):
    # Process text
    text = page.get_text()

    if text.strip():
        temp_doc = Document(
            page_content=text,
            metadata={"page": i, "type": "text"}
        )

        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    # Process embedded images
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            image_id = f"page_{i}_img_{img_index}"

            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")

            image_data_store[image_id] = base64.b64encode(
                buffered.getvalue()
            ).decode("utf-8")

            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={
                    "page": i,
                    "type": "image",
                    "image_id": image_id
                }
            )

            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")

doc.close()

print("Documents:", len(all_docs))
print("Embeddings:", len(all_embeddings))
print("Stored images:", len(image_data_store))

Documents: 6
Embeddings: 6
Stored images: 0


In [58]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='DOCRAG Test Document\nSynthetic sample for testing text retrieval, tables, and chart-image understanding\n1. Company Overview\nNorthstar Appliances is a fictional consumer-electronics company founded in 2018. It sells smart fans, air purifiers, and\nenergy-efficient kitchen devices through its website and retail partners. Its headquarters are in Pune, India. The company has\n240 employees as of December 2025.\n2. Quarterly Sales Summary'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='240 employees as of December 2025.\n2. Quarterly Sales Summary\nDuring the first half of 2025, sales increased gradually. The strongest quarter was Q3, when a regional retail partnership and a\nnew air-purifier launch contributed to higher demand. Q4 sales declined slightly from Q3, although they remained above Q1.\nQuarter\nUnits Sold\nRevenue (I lakh)\nCustomer Returns\nQ1 2025\n1,200\n36\n48\nQ2 2025\n1,450\n44\n52\nQ3 2025\n

In [ ]:
# # Create unified FAISS vector store with CLIP embeddings
# embeddings_array = np.array(all_embeddings)
# embeddings_array

array([], dtype=float64)

### Create the CLIP embedding adapter

In [59]:
from langchain_core.embeddings import Embeddings

class CLIPEmbeddings(Embeddings):
    """Use the existing CLIP functions with LangChain."""

    def embed_documents(self, texts):
        return [embed_text(text).tolist() for text in texts]

    def embed_query(self, text):
        return embed_text(text).tolist()


clip_embeddings = CLIPEmbeddings()

### Build the unified FAISS vector store

In [60]:
import faiss
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

# Convert precomputed CLIP vectors to FAISS-compatible format
embeddings_array = np.asarray(all_embeddings, dtype=np.float32)

assert len(all_docs) == len(embeddings_array), \
    "Documents and embeddings count mismatch!"

assert embeddings_array.ndim == 2 and embeddings_array.shape[1] == 512, \
    "Expected 512-dimensional embeddings!"

assert len(all_docs) > 0, "No documents to index!"

# CLIP embeddings are normalized, so inner product equals cosine similarity
faiss_index = faiss.IndexFlatIP(512)
faiss_index.add(embeddings_array)

# Preserve original documents, metadata and their ordering
docstore = InMemoryDocstore({
    str(i): doc for i, doc in enumerate(all_docs)
})

index_to_docstore_id = {
    i: str(i) for i in range(len(all_docs))
}

vector_store = FAISS(
    embedding_function=clip_embeddings,
    index=faiss_index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)

print("FAISS vector store created successfully!")
print("Indexed vectors:", vector_store.index.ntotal)

FAISS vector store created successfully!
Indexed vectors: 6


In [61]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-4.1")

print("GPT-4.1 initialized successfully!")

GPT-4.1 initialized successfully!


### Step 1: Create the multimodal RAG function
Run this cell after your existing FAISS index and llm initialization.

In [64]:
from langchain_core.messages import HumanMessage
import base64

def ask_pdf(question, k=5):
    """Answer questions using retrieved PDF text and images."""

    # 1. Retrieve relevant text and image references
    results = vector_store.similarity_search_with_score(
        question,
        k=min(k, len(all_docs))
    )

    if not results:
        return "No relevant content was found in the PDF."

    context_parts = []
    image_ids = []

    # 2. Separate text evidence from image references
    for doc, score in results:
        page = doc.metadata.get("page", "unknown")
        doc_type = doc.metadata.get("type")

        if doc_type == "text":
            context_parts.append(
                f"[Page {page+1} | Similarity {score:.3f}]\n"
                f"{doc.page_content}"
            )

        elif doc_type == "image":
            image_id = doc.metadata.get("image_id")
            if image_id and image_id in image_data_store:
                if image_id not in image_ids:
                    image_ids.append(image_id)

                context_parts.append(
                    f"[Page {page+1} | Image ID: {image_id} | "
                    f"Similarity {score:.3f}]"
                )

    # 3. Prepare the prompt
    context = "\n\n".join(context_parts)

    prompt = f"""
You are a document-grounded multimodal assistant.

Answer the user's question using the supplied PDF evidence.

Rules:
- Use the provided text and images as your primary evidence.
- Interpret charts, labels, legends, and visual trends when relevant.
- Combine textual and visual evidence when useful.
- Do not invent values or facts that cannot be verified.
- If the evidence is insufficient, explicitly say so.
- Mention the relevant page number when available.

Retrieved PDF evidence:
{context}

User question:
{question}
"""

    # 4. Send text and actual images to GPT-4.1
    message_content = [
        {"type": "text", "text": prompt}
    ]

    for image_id in image_ids:
        image_base64 = image_data_store[image_id]

        message_content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_base64}"
            }
        })

    message = HumanMessage(content=message_content)

    response = llm.invoke([message])

    return response.content

### Step 2: Test it with your revenue PDF
Try a question that requires both textual and visual understanding.

In [66]:
answer = ask_pdf(
    "Which quarter had the highest revenue, and how much was it? "
    "How many units were sold in Q2?"
    "Why did sales increase in Q3?"
    "What are the customer-support hours?"
    "Who is the CEO of Northstar Appliances?"
    "Compare the revenue trend from Q1 to Q4 using only the bar chart. Explain which quarter shows the peak and whether revenue rises or falls afterward. Do not use the text or table."
)

print(answer)

Here are the answers to your questions, using only the supplied PDF evidence:

1. **Which quarter had the highest revenue, and how much was it?**
   - The highest revenue was in Q3 2025, with ₹61 lakh.  
     (Evidence: Page 2, chart labeled "Quarterly Revenue — 2025" and supported by the Q&A section.)

2. **How many units were sold in Q2?**
   - 1,450 units were sold in Q2 2025.  
     (Evidence: Page 1, Quarterly Sales Summary table; Page 2, known facts section.)

3. **Why did sales increase in Q3?**
   - Sales increased in Q3 due to a regional retail partnership and the launch of a new air-purifier, which contributed to higher demand.  
     (Evidence: Page 1, Quarterly Sales Summary narrative.)

4. **What are the customer-support hours?**
   - Monday to Friday, 9:00 a.m. to 6:00 p.m. India Standard Time.  
     (Evidence: Page 1, Operating Policy section; Page 2, Q&A facts.)

5. **Who is the CEO of Northstar Appliances?**
   - The document does not provide information about the CEO

In [54]:
print(pdf_path)

multimodal_sample.pdf


### Then try a visual-specific question:

In [40]:
answer = ask_pdf(
    "Describe the chart in the PDF, including its labels, "
    "trend, and any numerical values that are clearly visible."
)

print(answer)

Based on the supplied chart from the PDF (page 0):

**Description of the Chart:**
- The chart is a vertical bar chart displaying three bars in sequence.
- Each bar is a different color: blue (first), green (second), and red (third).
- There are no visible numerical values, axis labels, or titles shown in the chart image itself.

**Trend:**
- The bars increase in height from left to right, indicating a positive upward trend.
- This visual trend supports the textual summary from the PDF, which states that revenue grew steadily across Q1 (blue), Q2 (green), and Q3 (red), with the highest revenue in Q3.

**Labels and Numerical Values:**
- There are no specific axis labels or numbers visible on the bars or axes in the image.
- However, the colors and order correlate with the textual evidence: blue for Q1, green for Q2, and red for Q3.

**Interpretation:**
- The chart visually supports the summary that Q1 had moderate growth, Q2 improved due to marketing, and Q3 experienced the most signific

In [37]:
query = "What is a large language model?"

results = vector_store.similarity_search_with_score(
    query,
    k=min(5, len(all_docs))
)

for doc, score in results:
    print("\nSimilarity score:", round(float(score), 4))
    print("Page:", doc.metadata.get("page"))
    print("Type:", doc.metadata.get("type"))
    print("Content:", doc.page_content[:300])


Similarity score: 0.5351
Page: 0
Type: text
Content: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart
below, revenue grew steadily with the highest growth recorded in Q3.
Q1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed
Q1 due to marketi

Similarity score: 0.2576
Page: 0
Type: image
Content: [Image: page_0_img_0]
